# Mecanismos de atención y auto-atención

En la clase 3.3 vimos que la **atención** reemplaza el estado de tamaño fijo de una RNN por un promedio ponderado de *valores*, con pesos que salen de la similitud entre una *consulta* y un conjunto de *claves*:

$$\mathrm{Attention}(\mathbf{Q},\mathbf{K},\mathbf{V})=\mathrm{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V}$$

En este laboratorio implementamos esa operación **desde cero en PyTorch**, en cuatro pasos de complejidad creciente:

1. Auto-atención simple, sin pesos entrenables.
2. Auto-atención con pesos entrenables $\mathbf{W}_q,\mathbf{W}_k,\mathbf{W}_v$.
3. Atención causal (el decodificador del Transformer y de los modelos tipo GPT).
4. Atención multi-cabeza.

Al final hay una **actividad** con ejercicios para completar.

> El desarrollo sigue el capítulo 3 de *Build a Large Language Model From Scratch* de Sebastian Raschka ([notebook original](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb), licencia Apache 2.0), traducido y adaptado a este curso.

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("torch version:", torch.__version__)

## 1. Auto-atención simple (sin pesos entrenables)

Usaremos una oración de seis palabras, donde cada palabra ya está representada por un embedding de 3 dimensiones (valores de ejemplo; en un modelo real tendrían cientos de dimensiones y se aprenderían).

En la **auto-atención** el objetivo es calcular, para cada elemento $\mathbf{x}^{(i)}$ de la entrada, un **vector de contexto** $\mathbf{z}^{(i)}$: una combinación de *todos* los elementos de la secuencia, ponderada por cuánta atención merece cada uno.

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89],  # Your     (x^1)
   [0.55, 0.87, 0.66],  # journey  (x^2)
   [0.57, 0.85, 0.64],  # starts   (x^3)
   [0.22, 0.58, 0.33],  # with     (x^4)
   [0.77, 0.25, 0.10],  # one      (x^5)
   [0.05, 0.80, 0.55]]  # step     (x^6)
)
palabras = ["Your", "journey", "starts", "with", "one", "step"]

### 1.1 Puntajes de atención para una consulta

Tomamos la segunda palabra (`journey`) como **consulta** $\mathbf{q}^{(2)}=\mathbf{x}^{(2)}$ y calculamos su puntaje contra cada palabra con el **producto punto**, que mide qué tan alineados están dos vectores.

In [ ]:
query = inputs[1]  # la 2ª palabra es la consulta

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

### 1.2 Normalización: de puntajes a pesos

Queremos pesos que sean no negativos y sumen 1 (una combinación convexa). La normalización más simple es dividir por la suma, pero en la práctica se usa **softmax**, que maneja mejor los valores extremos y tiene buenos gradientes.

In [ ]:
# Normalización simple: dividir por la suma
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Pesos:", attn_weights_2_tmp)
print("Suma:", attn_weights_2_tmp.sum())

In [ ]:
# Softmax "ingenuo": numéricamente inestable si hay valores grandes
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Pesos:", attn_weights_2_naive)
print("Suma:", attn_weights_2_naive.sum())

In [ ]:
# Softmax de PyTorch (estable numéricamente)
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Pesos:", attn_weights_2)
print("Suma:", attn_weights_2.sum())

### 1.3 Vector de contexto

El vector de contexto de `journey` es la suma de todas las palabras ponderada por los pesos de atención:

$$\mathbf{z}^{(2)}=\sum_{i=1}^{6}\alpha_{2i}\,\mathbf{x}^{(i)}$$

In [ ]:
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)

### 1.4 Todas las palabras a la vez

Repetir el cálculo para cada palabra equivale a operaciones matriciales: los puntajes son $\mathbf{X}\mathbf{X}^\top$ y los vectores de contexto son $\mathrm{softmax}(\mathbf{X}\mathbf{X}^\top)\,\mathbf{X}$. Primero con bucles (para ver qué ocurre) y luego en forma matricial.

In [ ]:
# Con bucles: matriz de 6x6, puntaje de cada palabra contra cada otra
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

In [ ]:
# Equivalente matricial (mucho más rápido)
attn_scores = inputs @ inputs.T
print(attn_scores)

In [ ]:
# softmax por filas: dim=-1 normaliza sobre las claves de cada consulta
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)
print("Suma de cada fila:", attn_weights.sum(dim=-1))

In [ ]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)
print("¿Coincide la fila 2 con el cálculo anterior?",
      torch.allclose(all_context_vecs[1], context_vec_2))

Para ver mejor qué palabras "miran" a cuáles, definimos una función auxiliar que dibuja la matriz de pesos como un mapa de calor. Cada **fila** es una consulta; cada **columna**, una clave.

In [ ]:
def plot_attention(weights, tokens, title="Pesos de atención", ax=None):
    # weights: tensor (n, n); filas = consultas, columnas = claves
    show = ax is None
    if ax is None:
        fig, ax = plt.subplots(figsize=(4.2, 4.2))
    im = ax.imshow(weights.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_xlabel("clave")
    ax.set_ylabel("consulta")
    ax.set_title(title)
    if show:
        plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        plt.show()

plot_attention(attn_weights, palabras, "Auto-atención simple")

**Observación.** Esta versión no tiene nada que aprender: la similitud depende solo de los embeddings de entrada. Además, cada palabra se atiende mucho a sí misma, porque $\mathbf{x}\cdot\mathbf{x}$ suele ser el producto punto más grande. En la siguiente sección agregamos parámetros entrenables para que el modelo decida *qué* similitud es útil.

## 2. Auto-atención con pesos entrenables

Ahora cada vector de entrada se proyecta de tres formas distintas:

$$\mathbf{q}^{(i)}=\mathbf{x}^{(i)}\mathbf{W}_q,\qquad \mathbf{k}^{(i)}=\mathbf{x}^{(i)}\mathbf{W}_k,\qquad \mathbf{v}^{(i)}=\mathbf{x}^{(i)}\mathbf{W}_v$$

con $\mathbf{W}_q,\mathbf{W}_k,\mathbf{W}_v\in\mathbb{R}^{d_\text{in}\times d_\text{out}}$ **aprendibles**. Esta es exactamente la formulación consulta--clave--valor de la clase.

### 2.1 Paso a paso para una sola palabra

In [ ]:
x_2 = inputs[1]        # segundo elemento de la entrada
d_in = inputs.shape[1]  # dimensión del embedding de entrada, d = 3
d_out = 2               # dimensión de salida (en GPT suele ser igual a d_in)

torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# requires_grad=False solo para simplificar las salidas; al entrenar, será True.

query_2 = x_2 @ W_query
print("query_2:", query_2)

In [ ]:
# Claves y valores para TODAS las palabras
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

In [ ]:
# Puntajes de la consulta 2 contra todas las claves
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

### 2.2 Por qué dividimos por $\sqrt{d_k}$

Si las componentes de $\mathbf{q}$ y $\mathbf{k}$ son independientes con media 0 y varianza 1, el producto $\mathbf{q}^\top\mathbf{k}$ tiene varianza $d_k$. Con $d_k$ grande, los puntajes tienen magnitudes enormes y el softmax **se satura** (un peso casi 1, el resto casi 0), con gradientes casi nulos. Dividir por $\sqrt{d_k}$ devuelve la varianza a 1. Lo comprobamos empíricamente:

In [ ]:
torch.manual_seed(0)
for d in [4, 64, 512]:
    q = torch.randn(10000, d)
    k = torch.randn(10000, d)
    s = (q * k).sum(dim=-1)
    print(f"d={d:4d}  var(q·k)={s.var():8.2f}   var(q·k/sqrt(d))={(s / math.sqrt(d)).var():5.2f}")

In [ ]:
# Efecto sobre el softmax: mismo vector de puntajes con y sin escalar (d = 512)
torch.manual_seed(1)
d = 512
q = torch.randn(d)
K = torch.randn(6, d)
s = K @ q
print("sin escalar :", torch.softmax(s, dim=-1))
print("con 1/sqrt(d):", torch.softmax(s / math.sqrt(d), dim=-1))

In [ ]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

### 2.3 Una clase compacta `SelfAttention`

Reunimos todo en un `nn.Module`. La primera versión usa `nn.Parameter` directamente.

In [ ]:
class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

La segunda versión usa `nn.Linear`, que (i) permite incluir un sesgo opcional y (ii) tiene una **inicialización de pesos mejor** que `torch.rand`, lo que estabiliza el entrenamiento. Es la forma estándar.

In [ ]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

## 3. Atención causal

En un modelo de lenguaje **autorregresivo** (decodificador del Transformer, GPT) la predicción del token $t$ solo puede depender de los tokens $\le t$. Si no lo impedimos, el modelo "haría trampa" mirando la palabra que debe predecir. La solución es la **máscara causal**: los puntajes de las posiciones futuras se reemplazan por $-\infty$ *antes* del softmax, de modo que su peso es exactamente 0.

### 3.1 Aplicar la máscara

Reutilizamos las proyecciones de `sa_v2` para obtener los pesos sin máscara:

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

**Enfoque ingenuo (no recomendado).** Multiplicar por una máscara triangular inferior y renormalizar. Funciona, pero hay una sutileza: los pesos se calcularon *con* el softmax que ya vio las posiciones futuras, así que hay que renormalizar a mano.

In [ ]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

masked_simple = attn_weights * mask_simple
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

**Enfoque eficiente.** Poner $-\infty$ en las posiciones futuras y *luego* aplicar softmax. Como $\exp(-\infty)=0$, el softmax ya produce filas normalizadas sin posiciones futuras, en un solo paso.

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)
print("¿Igual al enfoque ingenuo?", torch.allclose(attn_weights, masked_simple_norm))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
plot_attention(torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1), palabras, "Sin máscara", axs[0])
plot_attention(attn_weights, palabras, "Con máscara causal", axs[1])
plt.tight_layout()
plt.show()

### 3.2 Dropout sobre los pesos de atención

Para regularizar, durante el **entrenamiento** se puede aplicar *dropout* a los pesos de atención (después del softmax). Con tasa $p$, cada peso se anula con probabilidad $p$ y los restantes se reescalan por $1/(1-p)$ para conservar la magnitud esperada. En evaluación el dropout se desactiva.

In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)  # 50 % de los valores se anulan
example = torch.ones(6, 6)
print(dropout(example))

In [ ]:
torch.manual_seed(123)
print(dropout(attn_weights))

### 3.3 Clase `CausalAttention`

Agregamos las máscaras, el dropout y la dimensión de **lote** (batch). El tensor de máscara se registra con `register_buffer` para que se mueva automáticamente a la GPU junto con el modelo, sin ser un parámetro entrenable.

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)  # (2 ejemplos, 6 tokens, dimensión 3)

In [ ]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)   # transpone solo las dims de tokens/features
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        return attn_weights @ values

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, dropout=0.0)

context_vecs = ca(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

## 4. Atención multi-cabeza

Una sola cabeza produce un único patrón de atención. La **atención multi-cabeza** ejecuta varias cabezas en paralelo, cada una con sus propias proyecciones $\mathbf{W}_q,\mathbf{W}_k,\mathbf{W}_v$, de modo que cada una puede especializarse en una relación distinta (sintaxis, correferencia, posición relativa, etc.). Las salidas se concatenan y se proyectan con $\mathbf{W}_o$.

### 4.1 Versión ingenua: apilar cabezas

La forma más directa es crear varias instancias de `CausalAttention` y concatenar sus salidas.

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha_wrapper = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha_wrapper(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)  # la salida es num_heads * d_out = 4

Este enfoque recorre las cabezas **secuencialmente** con un bucle, lo cual desaprovecha la GPU.

### 4.2 Versión eficiente: dividir los pesos

En lugar de $h$ pares de matrices pequeñas, usamos **una** proyección grande y *dividimos* el resultado en $h$ cabezas con un `view`. Así todas las cabezas se calculan en una sola multiplicación de matrices por lote. Con `d_out` la dimensión total, cada cabeza trabaja con `head_dim = d_out // num_heads`.

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out debe ser divisible por num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads   # dimensión de cada cabeza

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)   # W_o: combina las cabezas
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)         # (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # -> (b, num_heads, num_tokens, head_dim): las cabezas actúan como dimensión de lote
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Atención escalada con máscara causal, en todas las cabezas a la vez
        attn_scores = queries @ keys.transpose(2, 3)          # (b, heads, T, T)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # (b, heads, T, head_dim) -> (b, T, heads, head_dim) -> (b, T, d_out)
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

### 4.3 Comprobando la multiplicación por lotes

El truco central es que `@` sobre tensores de 4 dimensiones multiplica matrices **independientemente** para cada cabeza. Lo verificamos con un tensor de forma `(b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)`.

In [ ]:
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

In [ ]:
first_head = a[0, 0, :, :]
second_head = a[0, 1, :, :]
print("Primera cabeza:\n", first_head @ first_head.T)
print("\nSegunda cabeza:\n", second_head @ second_head.T)

Las dos matrices coinciden con los dos bloques de la salida anterior: una sola operación calculó ambas cabezas.

### 4.4 Visualizando los patrones de cada cabeza

Para inspeccionar qué hace cada cabeza necesitamos sus pesos de atención. Escribimos una función que reproduce el cálculo de `MultiHeadAttention` y devuelve los pesos (antes de multiplicar por $\mathbf{V}$).

In [ ]:
@torch.no_grad()
def head_weights(mha, x):
    # Devuelve los pesos de atención de cada cabeza: (b, num_heads, T, T)
    b, T, _ = x.shape
    q = mha.W_query(x).view(b, T, mha.num_heads, mha.head_dim).transpose(1, 2)
    k = mha.W_key(x).view(b, T, mha.num_heads, mha.head_dim).transpose(1, 2)
    scores = q @ k.transpose(2, 3)
    scores = scores.masked_fill(mha.mask.bool()[:T, :T], -torch.inf)
    return torch.softmax(scores / mha.head_dim**0.5, dim=-1)

torch.manual_seed(42)
mha4 = MultiHeadAttention(d_in=3, d_out=8, context_length=6, dropout=0.0, num_heads=4)
w = head_weights(mha4, batch)   # (2, 4, 6, 6)

fig, axs = plt.subplots(1, 4, figsize=(13, 3.6))
for h in range(4):
    plot_attention(w[0, h], palabras, f"Cabeza {h + 1}", axs[h])
plt.tight_layout()
plt.show()

Con pesos aleatorios, las cabezas muestran patrones distintos solo porque tienen proyecciones distintas. En un modelo **entrenado**, cada cabeza se especializa en una relación útil para la tarea.

## Actividad

Complete las celdas marcadas con `# TU CÓDIGO AQUÍ`. Cada ejercicio tiene una celda de verificación que dice si el resultado es correcto (si aún no ha completado el ejercicio, solo le indicará que falta).

### Ejercicio 1: auto-atención a mano y el rol de la escala

Implemente la función `scaled_dot_product_attention(Q, K, V, mask=None)` que calcule
$$\mathrm{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$
y devuelva **la salida y los pesos**. Si `mask` es un tensor booleano, las posiciones donde vale `True` deben recibir peso 0 (use `masked_fill` con `-torch.inf`).

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # TU CÓDIGO AQUÍ
    # 1. puntajes = Q K^T / sqrt(d_k)
    # 2. si hay máscara, poner -inf donde mask sea True
    # 3. pesos = softmax por filas
    # 4. salida = pesos @ V
    raise NotImplementedError

In [ ]:
# Verificación del ejercicio 1
try:
    torch.manual_seed(0)
    Q, K, V = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
    causal = torch.triu(torch.ones(5, 5), diagonal=1).bool()

    out, w = scaled_dot_product_attention(Q, K, V, causal)
    ref = nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=True)

    assert torch.allclose(w.sum(-1), torch.ones(2, 5)), "las filas de pesos deben sumar 1"
    assert torch.allclose(w.triu(1), torch.zeros(2, 5, 5)), "no debe haber atención al futuro"
    assert torch.allclose(out, ref, atol=1e-5), "no coincide con F.scaled_dot_product_attention"
    print("Ejercicio 1 correcto")
except NotImplementedError:
    print("Ejercicio 1: pendiente")
except AssertionError as e:
    print("Ejercicio 1 incorrecto:", e)

### Ejercicio 2: comparar con la implementación de PyTorch

PyTorch incluye `nn.MultiheadAttention`. Cree una instancia con `embed_dim=8`, `num_heads=2`, `bias=False` y `batch_first=True`, y copie sus pesos dentro de nuestra clase `MultiHeadAttention` para comprobar que **calculan lo mismo**.

*Pista:* `nn.MultiheadAttention` guarda las tres proyecciones apiladas en `in_proj_weight` (de forma `(3*embed_dim, embed_dim)`, en orden $q,k,v$) y la salida en `out_proj.weight`. Divídala con `.chunk(3)` y copie cada parte con `.weight.data.copy_(...)`. Nuestra `out_proj` tiene sesgo; ponga el suyo en cero (`.bias.data.zero_()`).

In [ ]:
torch.manual_seed(7)
ref_mha = nn.MultiheadAttention(embed_dim=8, num_heads=2, bias=False, batch_first=True)
mine = MultiHeadAttention(d_in=8, d_out=8, context_length=5, dropout=0.0, num_heads=2)
x = torch.randn(3, 5, 8)

# TU CÓDIGO AQUÍ: copiar los pesos de ref_mha dentro de mine

In [ ]:
# Verificación del ejercicio 2
try:
    attn_mask = torch.triu(torch.ones(5, 5), diagonal=1).bool()   # True = no atender
    ref_out, _ = ref_mha(x, x, x, attn_mask=attn_mask, need_weights=False)
    my_out = mine(x)
    if torch.allclose(ref_out, my_out, atol=1e-5):
        print("Ejercicio 2 correcto: ambas implementaciones coinciden")
    else:
        print("Ejercicio 2: todavía no coinciden; error máximo =", (ref_out - my_out).abs().max().item())
except Exception as e:
    print("Ejercicio 2: pendiente o con error ->", type(e).__name__, e)

### Ejercicio 3: la auto-atención no conoce el orden

Demuestre que la auto-atención **sin** codificación posicional es *equivariante a permutaciones*: si se permutan los tokens de entrada, la salida es la misma permutación de la salida original. Use una atención **no causal** (`SelfAttention_v2`).

1. Cree una permutación `perm = torch.tensor([3, 0, 5, 1, 4, 2])`.
2. Calcule `y = sa_v2(inputs)` y `y_perm = sa_v2(inputs[perm])`.
3. Verifique con `torch.allclose` que `y_perm` es igual a `y[perm]`.

Luego implemente la **codificación posicional sinusoidal** de la clase,
$$p_{i,2j}=\sin\!\left(\frac{i}{10000^{2j/d}}\right),\qquad p_{i,2j+1}=\cos\!\left(\frac{i}{10000^{2j/d}}\right),$$
sume `P` a la entrada y compruebe que **ahora la igualdad se rompe**.

In [ ]:
def positional_encoding(n, d):
    # Devuelve una matriz P de forma (n, d). Suponga d par.
    # TU CÓDIGO AQUÍ
    raise NotImplementedError

perm = torch.tensor([3, 0, 5, 1, 4, 2])
# TU CÓDIGO AQUÍ: y, y_perm sin posición; y_pos, y_pos_perm con posición

In [ ]:
# Verificación del ejercicio 3
try:
    torch.manual_seed(0)
    emb = torch.randn(6, 4)
    sa = SelfAttention_v2(4, 4)
    P = positional_encoding(6, 4)
    assert P.shape == (6, 4), "P debe tener forma (n, d)"
    assert torch.allclose(P[0], torch.tensor([0., 1., 0., 1.])), "en i=0 se espera [sin 0, cos 0, sin 0, cos 0]"

    sin_pos = torch.allclose(sa(emb[perm]), sa(emb)[perm], atol=1e-5)
    con_pos = torch.allclose(sa(emb[perm] + P), sa(emb + P)[perm], atol=1e-5)
    print("sin posición, equivariante a permutaciones:", sin_pos, "(esperado: True)")
    print("con posición, equivariante a permutaciones:", con_pos, "(esperado: False)")
    print("Ejercicio 3 correcto" if sin_pos and not con_pos else "Ejercicio 3: revise su implementación")
except NotImplementedError:
    print("Ejercicio 3: pendiente")
except AssertionError as e:
    print("Ejercicio 3 incorrecto:", e)

> **Nota.** Con posición, la comparación correcta es `sa(emb[perm] + P)` contra `sa(emb + P)[perm]`: al reordenar la oración, cada token recibe la posición de su *nuevo* lugar, y por eso la salida deja de ser una simple permutación. Si en cambio se permutara también `P`, la posición viajaría pegada al token y no cambiaría nada.

### Ejercicio 4: un mini bloque Transformer

Combine lo visto con los demás componentes de la clase para armar un **bloque Transformer**:

$$\mathbf{Z}=\mathrm{LN}\big(\mathbf{X}+\mathrm{Drop}(\mathrm{MHA}(\mathbf{X}))\big),\qquad
\mathbf{Y}=\mathrm{LN}\big(\mathbf{Z}+\mathrm{Drop}(\mathrm{FFN}(\mathbf{Z}))\big)$$

con $\mathrm{FFN}(\mathbf{x})=\max(0,\mathbf{x}\mathbf{W}_1+\mathbf{b}_1)\mathbf{W}_2+\mathbf{b}_2$. Complete `TransformerBlock` usando `MultiHeadAttention` (con `context_length` suficientemente grande) y `nn.LayerNorm`. Nuestra `MultiHeadAttention` aplica máscara causal, así que el bloque resultante es el de un modelo tipo GPT (decodificador sin atención cruzada).

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, context_length, dropout=0.1):
        super().__init__()
        # TU CÓDIGO AQUÍ: self.attn, self.ffn (Linear-ReLU-Linear), self.norm1, self.norm2, self.drop
        raise NotImplementedError

    def forward(self, x):
        # TU CÓDIGO AQUÍ: dos subcapas con residual + dropout + LayerNorm
        raise NotImplementedError

In [ ]:
# Verificación del ejercicio 4
try:
    torch.manual_seed(0)
    block = TransformerBlock(d_model=16, num_heads=4, d_ff=32, context_length=10, dropout=0.0)
    x = torch.randn(2, 10, 16)
    y = block(x)
    assert y.shape == x.shape, "el bloque no debe cambiar la forma del tensor"
    # LayerNorm deja media ~0 y varianza ~1 sobre las características
    assert torch.allclose(y.mean(-1), torch.zeros(2, 10), atol=1e-5), "la salida no está normalizada"
    n_params = sum(p.numel() for p in block.parameters())
    print(f"Ejercicio 4 correcto: forma {tuple(y.shape)}, {n_params} parámetros")
except NotImplementedError:
    print("Ejercicio 4: pendiente")
except AssertionError as e:
    print("Ejercicio 4 incorrecto:", e)

### Preguntas de análisis

1. En la sección 1, ¿por qué cada palabra se atiende a sí misma con tanto peso? ¿Qué cambia al introducir $\mathbf{W}_q$ y $\mathbf{W}_k$ distintas?
2. En la sección 2.2, ¿qué sucede con el softmax para $d=512$ sin la escala $1/\sqrt{d}$? ¿Cómo afectaría esto al gradiente?
3. ¿Por qué la máscara se aplica con $-\infty$ *antes* del softmax y no multiplicando los pesos *después*?
4. Con `d_out` fijo, ¿qué ocurre con el costo de cómputo y de parámetros al aumentar `num_heads`? ¿Qué se pierde si `head_dim` es muy pequeño?
5. La auto-atención tiene costo $\mathcal{O}(n^2 d)$. Estime cuántas entradas tiene la matriz de pesos para un contexto de 8\,000 tokens y comente las implicancias para la memoria.

### Referencias

- Raschka, S. *Build a Large Language Model From Scratch*, cap. 3. [github.com/rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch)
- Zhang et al. *Dive into Deep Learning*, cap. 11: [d2l.ai/chapter_attention-mechanisms-and-transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- Vaswani et al. (2017). *Attention is all you need*.